## CSC 380 HW#4: LLM and RAG -- **Starter Code**

Spring 2026 HW#4 — fill in every `TODO` block and run all cells.


## Write your name here.

In [ ]:
# Setup  (do not modify)
# ----------------------------
# Colab already has torch, transformers, and sentence-transformers pre-installed.
# Only install the LangChain ecosystem packages that are not pre-installed.
# Do NOT use -U or --force-reinstall here — that breaks Colab's torch/transformers.
!pip install -q \
    langchain langchain-core langchain-community \
    langchain-huggingface langchain-openai langchain-text-splitters \
    faiss-cpu openai tiktoken pypdf sentence-transformers

In [ ]:
# Imports  (do not modify)
# ----------------------------
import os, re, json as _json, time
import requests
from typing import List, Dict, Tuple
from IPython.display import Markdown, display

from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [ ]:
# Load API keys from CoLab secret keys  (do not modify)
from google.colab import userdata

langchain_api = userdata.get('langchain_api')
openai_api    = userdata.get('openai_api')
hf_token      = userdata.get('HF_TOKEN')

os.environ['OPENAI_API_KEY']           = openai_api
os.environ['LANGCHAIN_API_KEY']        = langchain_api
os.environ['HUGGINGFACEHUB_API_TOKEN'] = hf_token

## Part 1: Define Core Functions

In [ ]:
# @title
# 1-1: Load documents  (provided — do not modify)
# ----------------------------
def load_documents() -> List[Document]:
    """
    Load the nutrition / diabetes documents from GitHub.
    Returns a list of Document objects.
    """
    url = "https://raw.githubusercontent.com/ntomuro/CSC380/main/HW4-LLM_RAG/data"
    document_filenames = [
        "Human-Nutrition-2020-Edition-1598491699.txt",
        "dci190009_pdf.txt",
        "dci190014_pdf.txt"
    ]
    documents = []
    for doc in document_filenames:
        document_url = f"{url}/{doc}"
        try:
            response = requests.get(document_url)
            response.raise_for_status()
            documents.append(Document(page_content=response.text,
                                      metadata={"source": document_url}))
        except requests.exceptions.RequestException as e:
            print(f"Error loading {document_url}: {e}")
    return documents


# 1-2: Preprocess documents
# ----------------------------
def preprocess_documents(documents: List[Document]) -> List[Document]:
    """
    Split documents into smaller chunks for vector storage.
    Returns a list of Document chunks.
    """
    # TODO (1): Create a text splitter and split the documents into chunks.
    #
    # Hint: Use RecursiveCharacterTextSplitter(chunk_size=..., chunk_overlap=...)
    # Hint: Call text_splitter.split_documents(documents) to get the chunks.
    pass  # replace with your code


# 1-3: Vector Store Population
# ----------------------------
def create_vector_store(documents: List[Document], embedding_model_name: str):
    """
    Build a FAISS vector store from document chunks and return a retriever.
    FAISS is in-memory — no disk writes, no permission issues.
    Returns a retriever object.
    """
    # TODO (2): Implement vector store creation in two steps.
    # Step 1 — Create an embedding function:
    # Step 2 — Build the FAISS vector store and return a retriever:
    pass  # replace with your code


# 1-4: Query & Evaluation Set
# ----------------------------
def create_test_queries() -> List[str]:
    """
    Return a list of 8 test queries of your own design.

    Requirements:
      - Queries 1-4: answerable from the stored documents
          (diabetes, nutrition, prediabetes topics).
      - Queries 5-8: NOT covered by the stored documents.
          These test whether the model admits it doesn't know
          rather than hallucinating an answer.
    """
    # TODO (3): Write your own 8 queries.
    #
    # Hint: Good queries for the first four ask about something specific
    #       in the nutrition/diabetes texts (e.g., a nutrient, a risk factor,
    #       a recommended diet). Good queries for the last four sound plausible
    #       but cover material absent from these documents (e.g., a specific
    #       medication, a clinical trial, or a topic outside the documents' scope).
    queries = [
        "",   # answerable — factual
        "",   # answerable — factual
        "",   # answerable — broader / synthesis
        "",   # answerable — broader / synthesis
        "",   # unanswerable — near-answerable but not
        "",   # unanswerable — near-answerable but not
        "",   # unanswerable — tests hallucination
        "",   # unanswerable — tests hallucination
    ]
    return queries


# 1-5: The RAG Chain  (modern LCEL style)
# ----------------------------
def run_rag_query(query: str, retriever) -> Tuple[str, List[Document]]:
    """
    Execute a RAG query and return (answer, retrieved_docs).

    The pipeline follows the LCEL (LangChain Expression Language) pattern:
        inputs -> prompt -> LLM -> string output
    """
    # Step 1 — Retrieve relevant chunks.
    # TODO  (4): Invoke the retriever with the querto get a list of Document objects.
    retrieved_docs = None  # replace with your code

    # Step 2 — Define a prompt template.
    # TODO (5): Use PromptTemplate.from_template(...) to create a RAG prompt that:
    #   • includes a {context} variable (the retrieved text) and a {question} variable.
    #   • instructs the model to answer from the context only and say clearly
    #     "I don't know" or something if the answer isn't there.
    prompt = None  # replace with your code

    # Step 3 — Initialize the LLM (to perform RAG).
    # TODO (6): Create an LLM (model="gpt-4o-mini") with temperature 0.1.
    llm = None  # replace with your code

    def format_docs(docs):   # helper — joins chunk text with blank lines
        return "\n\n".join(d.page_content for d in docs)

    # Step 4 — Build the LCEL chain with | (pipe) operators.
    # TODO (7): Assemble the chain -- process pipeline, consisting of
    #  1. concatenated retrieved docs
    #  2. RAG prompt
    #  3. LLM (the RAG model)
    #  4. output parser
    chain = None  # replace with your code

    # Step 5 — Run the chain.
    # TODO (8): Invoke the chain with query and get the answer string.
    answer = None  # replace with your code

    return answer, retrieved_docs


# 1-6: LLM-as-Judge Evaluation  (agentic self-evaluation)
# ----------------------------
def evaluate_results(query: str, retrieved_docs: List[Document], answer: str) -> Tuple[int, int]:
    """
    Use a *second* LLM call to evaluate the RAG system's output.
    Returns (retrieval_score, answer_score) each on a 1-5 scale.

    This is an agentic pattern: instead of a human scoring every response,
    an LLM judge critiques the system automatically.

    Scores:
      retrieval_score: 1=irrelevant,.. YOU DECIDE (up to 5=..)
      answer_score:    1=incorrect,  YOU DECIDE (up to 5=..)
    """
    # TODO (9): Build an LLM-as-judge chain in five steps.
    #
    # Step 1 — Write an eval_prompt using PromptTemplate.from_template(...).
    #   Include variables: {query}, {context}, {answer}.
    #   Ask the LLM to score retrieval (1-5) and answer quality (1-5).
    #   Tell it to respond with ONLY this JSON and nothing else:
    #     {"retrieval_score": <int>, "answer_score": <int>, "reason": "<one sentence>"}
    #   Hint: to include literal braces in a Python template string, double them: {{ }}
    #
    # Step 2 — Create the judge LLM (model="gpt-4o-mini") with temperature=0.
    #
    # Step 3 — Build the chain ('judge chain'): eval_prompt | llm | StrOutputParser()
    #
    # Step 4 — Invoke the chain. Pass:
    #   context = retrieved_docs[0].page_content[:500]  (first 500 chars of top chunk)
    #   query   = query
    #   answer  = answer
    #
    # Step 5 — Parse the JSON string and return two scores -- retrieval score and answer score.
    #   Use a try/except so a bad LLM response doesn't crash the whole experiment:
    pass  # replace with your code

## Part 2: Experiment (main())

In [ ]:
# @title
# 2: The Main Experiment  (provided — do not modify)
# ----------------------------
def main():
    """
    Run the complete RAG experiment across two embedding models.
    """
    embedding_models = [
        "all-MiniLM-L6-v2",
        "multi-qa-MiniLM-L6-cos-v1",
    ]

    print("Loading documents...")
    raw_documents = load_documents()
    print(f"Loaded {len(raw_documents)} raw documents")

    print("Preprocessing documents...")
    processed_documents = preprocess_documents(raw_documents)
    print(f"Created {len(processed_documents)} document chunks")

    test_queries = create_test_queries()
    print(f"Created {len(test_queries)} test queries")

    results = {}

    for model_name in embedding_models:
        print(f"\n{'='*50}")
        print(f"Testing model: {model_name}")
        print(f"{'='*50}")

        retriever = create_vector_store(processed_documents, model_name)

        # Step 1: run all queries and store results in memory
        rag_results = []
        for i, query in enumerate(test_queries):
            print(f"\nQuery {i+1}: {query}")
            answer, retrieved_docs = run_rag_query(query, retriever)
            rag_results.append({"query": query, "answer": answer, "retrieved_docs": retrieved_docs})
            print(f"Answer: {answer}")
            time.sleep(2)  # avoid OpenAI rate limits

        # Step 2: evaluate stored results — LLM as judge
        # Print LLM-as-judge results
        print ("\n------- LLM as Judge ---------")

        model_results = {
            'retrieval_scores': [],
            'answer_scores':    [],
            'details':          []
        }
        for i, r in enumerate(rag_results):
            #retrieval_score, answer_score = evaluate_results(r["query"], r["retrieved_docs"], r["answer"])
            scores = evaluate_results(r["query"], r["retrieved_docs"], r["answer"])
            retrieval_score = scores["retrieval_score"]
            answer_score = scores["answer_score"]
            print(f"\nQuery {i+1}:")
            print (f"Retrieval Score: {retrieval_score}, Answer Score: {answer_score}, Reason: {scores['reason']}")

            model_results['retrieval_scores'].append(retrieval_score)
            model_results['answer_scores'].append(answer_score)
            model_results['details'].append({
                'query':          r["query"],
                'answer':         r["answer"],
                'retrieved_docs': r["retrieved_docs"]
            })
            time.sleep(2)

        model_results['avg_retrieval'] = (
            sum(model_results['retrieval_scores']) / len(model_results['retrieval_scores'])
        )
        model_results['avg_answer'] = (
            sum(model_results['answer_scores']) / len(model_results['answer_scores'])
        )
        results[model_name] = model_results

        print(f"\nModel {model_name} — "
              f"Avg Retrieval: {model_results['avg_retrieval']:.2f}, "
              f"Avg Answer: {model_results['avg_answer']:.2f}")

    return results, processed_documents, test_queries

## Part 3: Run Experiment

In [ ]:
# @title
# Execute the experiment  (run after completing all TODOs)
final_results, documents, queries = main()
print(final_results)

### Print Results

In [ ]:
# Analysis Helper Functions  (provided — do not modify)
# ----------------------------

def print_results_table(results: Dict):
    """Print a formatted results table for the report."""
    print("\n" + "="*60)
    print("EXPERIMENT RESULTS SUMMARY")
    print("="*60)
    print(f"{'Model':<25} {'Avg Retrieval':<15} {'Avg Answer':<15}")
    print("-"*60)
    for model_name, model_results in results.items():
        print(f"{model_name:<25} "
              f"{model_results['avg_retrieval']:<15.2f} "
              f"{model_results['avg_answer']:<15.2f}")


def compare_retrieval_for_query(results: Dict, query_index: int):
    """Compare retrieval performance for a specific query across models."""
    print(f"\nRetrieval comparison for Query {query_index + 1}:")
    for model_name, model_results in results.items():
        r = model_results['retrieval_scores'][query_index]
        a = model_results['answer_scores'][query_index]
        print(f"  {model_name}: Retrieval={r}, Answer={a}")


# Print the final results
print_results_table(final_results)
for i, query in enumerate(queries):
    compare_retrieval_for_query(final_results, i)